# 🔍 Search Algorithms – Programming Exercises
**Foundations of Artificial Intelligence · SS 2026 · University of Bremen**

---

In this notebook you will implement BFS, DFS, UCS and A* step by step.

**No prior programming experience needed!**  
Each exercise is self-contained and comes with plenty of hints. Just fill in the parts marked with `# YOUR CODE HERE`.

Run a cell with **Shift + Enter**.

## 🗺️ The Graph we will use

```
        S
       / \
    1 /   \ 3
     /     \
    A       B
     \     / \
    3 \   / 1 \ 2
       \ /     \
        C       G
         \
         12\
             G
```

Edges:
- S → A (cost 1), S → B (cost 3)
- A → C (cost 3)
- B → C (cost 1), B → G (cost 2)
- C → G (cost 12)

Heuristic estimates h(n):
- h(S)=4, h(A)=3, h(B)=2, h(C)=2, h(G)=0

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
# Run this cell first! It defines the graph data we will use in all exercises.

# Unweighted graph (for BFS / DFS)
# Format: node -> list of neighbors
unweighted_graph = {
    'S': ['A', 'B'],
    'A': ['S', 'C'],
    'B': ['S', 'C', 'G'],
    'C': ['A', 'B', 'G'],
    'G': []
}

# Weighted graph (for UCS / A*)
# Format: node -> list of (neighbor, cost) tuples
weighted_graph = {
    'S': [('A', 1), ('B', 3)],
    'A': [('S', 1), ('C', 3)],
    'B': [('S', 3), ('C', 1), ('G', 2)],
    'C': [('A', 3), ('B', 1), ('G', 12)],
    'G': []
}

# Heuristic values
heuristic = {
    'S': 4,
    'A': 3,
    'B': 2,
    'C': 2,
    'G': 0
}

START = 'S'
GOAL  = 'G'

print("Setup complete ✅")

---
## Exercise 3 – Breadth-First Search (BFS) and Depth-First Search (DFS)

### How BFS works
BFS explores the graph **level by level** (closest nodes first).  
It uses a **queue** (First-In First-Out): we always take the **oldest** path from the front.

```
queue = [ [S] ]
take [S]  → expand S → add [S,A] and [S,B] to the END
queue = [ [S,A], [S,B] ]
take [S,A] → expand A → ...
```

### How DFS works
DFS explores the graph **depth first** (goes as deep as possible before backtracking).  
It uses a **stack** (Last-In First-Out): we always take the **newest** path from the top.

In [ ]:
from collections import deque   # deque is a fast queue

def bfs(graph, start, goal):
    """
    Breadth-First Search.
    Returns the path from start to goal as a list of node names,
    and the number of nodes visited.
    """
    # The queue holds PATHS (lists of nodes), not just single nodes.
    # We start with a queue containing only the path [start].
    queue   = deque([[start]])
    visited = set()             # keeps track of already-seen nodes
    visited.add(start)
    nodes_visited = 0

    while queue:
        # Take the FIRST path from the queue  (oldest = front)
        path = queue.popleft()
        node = path[-1]         # the last node in the current path
        nodes_visited += 1

        # ── Goal check ──────────────────────────────────────────────────
        if node == goal:
            return path, nodes_visited

        # ── Expand: look at all neighbors ───────────────────────────────
        for neighbor in graph.get(node, []):
            if neighbor not in visited:
                visited.add(neighbor)
                # YOUR CODE HERE:
                # Add the new path (path + [neighbor]) to the END of the queue
                # Hint: queue.append(  ???  )
                pass

    return None, nodes_visited  # no path found


# ── Test ────────────────────────────────────────────────────────────────────
path_bfs, visited_bfs = bfs(unweighted_graph, START, GOAL)
print(f"BFS path:          {path_bfs}")
print(f"BFS nodes visited: {visited_bfs}")

In [ ]:
def dfs(graph, start, goal):
    """
    Depth-First Search.
    Returns the path from start to goal and the number of nodes visited.
    """
    # A stack also holds PATHS. We use a regular Python list as a stack.
    stack   = [[start]]
    visited = set()
    visited.add(start)
    nodes_visited = 0

    while stack:
        # YOUR CODE HERE:
        # Take the LAST path from the stack (newest = top)
        # Hint: stack.pop()  removes and returns the last element
        path = None  # replace None with the correct call
        node = path[-1]
        nodes_visited += 1

        if node == goal:
            return path, nodes_visited

        for neighbor in graph.get(node, []):
            if neighbor not in visited:
                visited.add(neighbor)
                # YOUR CODE HERE:
                # Add the new path to the stack
                pass

    return None, nodes_visited


# ── Test ────────────────────────────────────────────────────────────────────
path_dfs, visited_dfs = dfs(unweighted_graph, START, GOAL)
print(f"DFS path:          {path_dfs}")
print(f"DFS nodes visited: {visited_dfs}")
print()
print("Are the paths the same?", path_bfs == path_dfs)

**✏️ Question 3.3** – Are the paths BFS and DFS found the same?  
Write your answer below (double-click to edit):

> *Your answer here...*

---
## Exercise 4 – Uniform Cost Search (UCS)

UCS is like BFS but uses **edge costs**. Instead of a normal queue it uses a **priority queue**:  
the path with the **lowest total cost so far** is always expanded next.

Python's `heapq` module gives us an efficient priority queue:
```python
import heapq
pq = []
heapq.heappush(pq, (priority, item))   # add item with given priority
priority, item = heapq.heappop(pq)     # remove and return the item with LOWEST priority
```

In [ ]:
import heapq

def ucs(graph, start, goal):
    """
    Uniform Cost Search.
    Returns (path, total_cost, nodes_visited).
    """
    # Priority queue entries: (cost_so_far, node, path)
    pq = [(0, start, [start])]
    visited = set()
    nodes_visited = 0

    while pq:
        # Take the entry with the LOWEST cost
        cost, node, path = heapq.heappop(pq)

        # Skip if we already found a cheaper way to this node
        if node in visited:
            continue
        visited.add(node)
        nodes_visited += 1

        if node == goal:
            return path, cost, nodes_visited

        # Expand neighbors
        for neighbor, edge_cost in graph.get(node, []):
            if neighbor not in visited:
                new_cost = cost + edge_cost
                # YOUR CODE HERE:
                # Push (new_cost, neighbor, path + [neighbor]) onto the priority queue
                # Hint: heapq.heappush(pq, (???, ???, ???))
                pass

    return None, float('inf'), nodes_visited


# ── Test ────────────────────────────────────────────────────────────────────
path_ucs, cost_ucs, visited_ucs = ucs(weighted_graph, START, GOAL)
print(f"UCS path:          {path_ucs}")
print(f"UCS total cost:    {cost_ucs}")
print(f"UCS nodes visited: {visited_ucs}")
print()
print("Does this match your hand-trace from Part 2?")

---
## Exercise 5 – A* Search

A* is like UCS but adds a **heuristic** to guide the search:

$$f(n) = g(n) + h(n)$$

- `g(n)` = cost from the start to node `n` (same as UCS)
- `h(n)` = estimated cost from `n` to the goal (the heuristic)

The priority queue is sorted by `f(n)` instead of just `g(n)`.

In [ ]:
import heapq

def astar(graph, h, start, goal):
    """
    A* Search.
    h  : dictionary mapping node -> heuristic estimate
    Returns (path, total_cost, nodes_visited).
    """
    # Priority queue entries: (f_value, g_value, node, path)
    pq = [(h[start], 0, start, [start])]
    visited = set()
    nodes_visited = 0

    while pq:
        f, g, node, path = heapq.heappop(pq)

        if node in visited:
            continue
        visited.add(node)
        nodes_visited += 1

        if node == goal:
            return path, g, nodes_visited

        for neighbor, edge_cost in graph.get(node, []):
            if neighbor not in visited:
                new_g = g + edge_cost
                # YOUR CODE HERE:
                # 1. Calculate new_f = new_g + h[neighbor]
                # 2. Push (new_f, new_g, neighbor, path + [neighbor]) onto pq
                pass

    return None, float('inf'), nodes_visited


# ── Test 5.2 ────────────────────────────────────────────────────────────────
path_as, cost_as, visited_as = astar(weighted_graph, heuristic, START, GOAL)
print(f"A* path:          {path_as}")
print(f"A* total cost:    {cost_as}")
print(f"A* nodes visited: {visited_as}")
print()
print(f"UCS nodes visited: {visited_ucs}")
print("A* visits fewer nodes:", visited_as < visited_ucs)

In [ ]:
# ── Bonus 5.3 ───────────────────────────────────────────────────────────────
# Change h(C) = 12 and run A* again.

heuristic_modified = dict(heuristic)   # copy so we don't break the original
heuristic_modified['C'] = 12

path_bonus, cost_bonus, _ = astar(weighted_graph, heuristic_modified, START, GOAL)
print(f"A* (h_C=12) path:  {path_bonus}")
print(f"A* (h_C=12) cost:  {cost_bonus}")
print()
print("Is it still optimal (cost = 5)?", cost_bonus == 5)

**✏️ Question 5.3** – Why does A\* still find the optimal solution even with h(C)=12?  
Is the heuristic still admissible?

> *Your answer here...*

---
## 📊 Summary: Compare all algorithms

In [ ]:
# Run this cell after completing all exercises to see a summary table.
print(f"{'Algorithm':<10} {'Path':<25} {'Cost':<8} {'Nodes visited':<15}")
print("-" * 60)
print(f"{'BFS':<10} {str(path_bfs):<25} {'N/A':<8} {visited_bfs:<15}")
print(f"{'DFS':<10} {str(path_dfs):<25} {'N/A':<8} {visited_dfs:<15}")
print(f"{'UCS':<10} {str(path_ucs):<25} {cost_ucs:<8} {visited_ucs:<15}")
print(f"{'A*':<10} {str(path_as):<25} {cost_as:<8} {visited_as:<15}")

---
## ✏️ Part 6 – Reflection Questions

Answer each question in 2–3 sentences by double-clicking this cell.

**6.1** What is the difference between an *admissible* and a *consistent* heuristic?

> *Your answer...*

**6.2** When would you prefer A\* over UCS?

> *Your answer...*

**6.3** Why does graph-search A\* need a *consistent* heuristic, while tree-search A\* only needs *admissibility*?

> *Your answer...*

---
🎓 **Well done!** You have implemented four fundamental search algorithms from scratch.